In [2]:
import os
import json
import pandas as pd
import numpy as np

# Configuration paths
INPUT_FILE = r"C:\Users\Edumo\Desktop\Milwaukee Crime Analysis\milwaukee_crime_processed.geojson"
OUTPUT_DIR = r"C:\Users\Edumo\Desktop\Milwaukee Crime Analysis"

def run_omap_analysis():
    print("Initializing Milwaukee Police Department OMAP Data Pipeline...")
    
    if not os.path.exists(INPUT_FILE):
        print(f"Error: Could not locate input file at {INPUT_FILE}")
        return

    # Load dataset
    with open(INPUT_FILE, 'r') as f:
        geojson_data = json.load(f)
    
    features = [feat['properties'] for feat in geojson_data['features']]
    df = pd.DataFrame(features)
    df.columns = [c.lower() for c in df.columns]
    
    print(f"Dataset loaded successfully. Total records: {len(df):,}")
    print(f"Available columns: {list(df.columns)}")

    # Automatically find the date/time column if it's named differently
    potential_time_cols = ['timestamp', 'incident_date', 'date', 'datetime', 'date_time', 'time']
    time_col = next((c for c in potential_time_cols if c in df.columns), None)
    
    # Fallback: search for any column containing 'date' or 'time'
    if not time_col:
        time_col = next((c for c in df.columns if 'date' in c or 'time' in c), None)

    if time_col:
        print(f"Using datetime column: '{time_col}'")
        df['parsed_timestamp'] = pd.to_datetime(df[time_col], errors='coerce')
        df['hour'] = df['parsed_timestamp'].dt.hour
        df['day_of_week'] = df['parsed_timestamp'].dt.day_name()
        df['year_month'] = df['parsed_timestamp'].dt.to_period('M')
    else:
        raise ValueError("Could not find a valid date/time column in the GeoJSON properties.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Hourly & Shift Workload Breakdown
    print("\n[1/3] Processing shift allocation and hourly demand...")
    
    def assign_shift(hour):
        if pd.isna(hour):
            return "Unknown"
        elif 8 <= hour < 16:
            return "1st Shift (Day)"
        elif 16 <= hour < 24:
            return "2nd Shift (Evening)"
        else:
            return "3rd Shift (Night)"

    df['operational_shift'] = df['hour'].apply(assign_shift)
    
    shift_summary = df['operational_shift'].value_counts().reset_index()
    shift_summary.columns = ['Operational_Shift', 'Shift_Incidents']
    total_incidents = len(df)
    shift_summary['Workload_Pct'] = (shift_summary['Shift_Incidents'] / total_incidents)
    shift_summary['Recommended_FTE'] = np.round(shift_summary['Workload_Pct'] * 100).astype(int)
    
    hourly_summary = df['hour'].value_counts().sort_index().reset_index()
    hourly_summary.columns = ['Hour_of_Day', 'Incident_Count']
    hourly_summary['Pct_Total_Volume'] = (hourly_summary['Incident_Count'] / total_incidents) * 100

    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_summary = df['day_of_week'].value_counts().reindex(day_order).reset_index()
    day_summary.columns = ['Day_of_Week', 'Incident_Count']
    day_summary['Pct_Total'] = (day_summary['Incident_Count'] / total_incidents) * 100

    # 2. Time-Series & 3-Month Moving Average Forecasting
    print("[2/3] Calculating monthly trends and moving averages...")
    monthly_series = df.groupby('year_month').size().reset_index(name='Monthly_Incidents')
    monthly_series['year_month'] = monthly_series['year_month'].astype(str)
    monthly_series['3_Month_MA'] = monthly_series['Monthly_Incidents'].rolling(window=3).mean().round(1)

    # 3. Exporting All Outputs to Desktop
    print("[3/3] Exporting processed tables to CSV...")
    shift_summary.to_csv(os.path.join(OUTPUT_DIR, 'shift_allocation_model.csv'), index=False)
    hourly_summary.to_csv(os.path.join(OUTPUT_DIR, 'hourly_workload_model.csv'), index=False)
    day_summary.to_csv(os.path.join(OUTPUT_DIR, 'day_workload_model.csv'), index=False)
    monthly_series.to_csv(os.path.join(OUTPUT_DIR, 'monthly_time_series_model.csv'), index=False)

    print(f"\nPipeline complete! All consolidated output files saved to:\n  {OUTPUT_DIR}")

if __name__ == "__main__":
    run_omap_analysis()

Initializing Milwaukee Police Department OMAP Data Pipeline...
Dataset loaded successfully. Total records: 112,283
Available columns: ['timestamp', 'year', 'month', 'dayofweek', 'hour', 'operationalshift']
Using datetime column: 'timestamp'

[1/3] Processing shift allocation and hourly demand...
[2/3] Calculating monthly trends and moving averages...
[3/3] Exporting processed tables to CSV...

Pipeline complete! All consolidated output files saved to:
  C:\Users\Edumo\Desktop\Milwaukee Crime Analysis
